# GOLD LAYER

In [0]:
# ============================================================
# 07_GOLD
#
# Purpose:
# Create business-ready analytical tables
#
# Input:
# ecommerce.silver
#
# Output:
# ecommerce.gold
#
# Concepts:
# - Joins
# - Aggregations
# - Business KPIs
# - Delta Tables
#
# ============================================================


from pyspark.sql.functions import *



# ============================================================
# 1. Configuration
# ============================================================


catalog = "ecommerce"

silver_schema = "silver"

gold_schema = "gold"



# ============================================================
# 2. Create Gold Schema
# ============================================================


spark.sql("""
CREATE SCHEMA IF NOT EXISTS ecommerce.gold
""")


print("Gold schema ready")



# ============================================================
# 3. Read Silver Tables
# ============================================================


customers = spark.table(
    "ecommerce.silver.customers"
)


products = spark.table(
    "ecommerce.silver.products"
)


orders = spark.table(
    "ecommerce.silver.orders"
)


payments = spark.table(
    "ecommerce.silver.payments"
)


returns = spark.table(
    "ecommerce.silver.returns"
)





# ============================================================
# GOLD TABLE 1
#
# Monthly Sales Analysis
#
# Business Question:
# How much revenue did we generate each month?
#
# ============================================================


monthly_sales = (

    orders


    .withColumn(
        "month",
        date_format(
            col("order_date"),
            "yyyy-MM"
        )
    )


    .groupBy(
        "month"
    )


    .agg(

        sum("total_amount")
        .alias("total_revenue"),


        countDistinct("order_id")
        .alias("total_orders"),


        avg("total_amount")
        .alias("average_order_value")

    )


)



monthly_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.monthly_sales"
)



print("Monthly Sales created")





# ============================================================
# GOLD TABLE 2
#
# Customer Summary
#
# Business Question:
# Who are our most valuable customers?
#
# ============================================================


customer_summary = (

    orders


    .groupBy(
        "customer_id"
    )


    .agg(

        countDistinct("order_id")
        .alias("total_orders"),


        sum("total_amount")
        .alias("total_spent"),


        avg("total_amount")
        .alias("average_order_value")

    )


    .join(

        customers,

        "customer_id",

        "left"

    )


    .select(

        "customer_id",

        "first_name",

        "last_name",

        "country",

        "membership",

        "total_orders",

        "total_spent",

        "average_order_value"

    )

)



customer_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.customer_summary"
)



print("Customer Summary created")





# ============================================================
# GOLD TABLE 3
#
# Product Performance
#
# Business Question:
# Which products generate most revenue?
#
# ============================================================


product_performance = (

    orders


    .groupBy(
        "product_id"
    )


    .agg(

        sum("quantity")
        .alias("units_sold"),


        sum("total_amount")
        .alias("revenue"),


        countDistinct("order_id")
        .alias("number_of_orders")

    )


    .join(

        products,

        "product_id",

        "left"

    )


    .select(

        "product_id",

        "product_name",

        "category",

        "brand",

        "units_sold",

        "revenue",

        "number_of_orders"

    )

)



product_performance.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.product_performance"
)



print("Product Performance created")





# ============================================================
# GOLD TABLE 4
#
# Sales By Country
#
# Business Question:
# Which countries generate the most sales?
#
# ============================================================


sales_by_country = (

    orders


    .groupBy(
        "country"
    )


    .agg(

        sum("total_amount")
        .alias("total_sales"),


        countDistinct("order_id")
        .alias("orders")

    )


)



sales_by_country.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.sales_by_country"
)



print("Sales By Country created")





# ============================================================
# GOLD TABLE 5
#
# Return Analysis
#
# Business Question:
# Which products have high return rates?
#
# ============================================================


return_analysis = (

    returns


    .groupBy(
        "product_id",
        "return_reason"
    )


    .agg(

        count("*")
        .alias("total_returns"),


        sum("refund_amount")
        .alias("total_refunded")

    )


    .join(

        products,

        "product_id",

        "left"

    )


    .select(

        "product_id",

        "product_name",

        "category",

        "return_reason",

        "total_returns",

        "total_refunded"

    )

)



return_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.return_analysis"
)



print("Return Analysis created")





# ============================================================
# GOLD TABLE 6
#
# Payment Analysis
#
# Business Question:
# Which payment methods are popular?
#
# ============================================================


payment_analysis = (

    payments


    .groupBy(
        "payment_method"
    )


    .agg(

        count("*")
        .alias("transactions"),


        sum("amount")
        .alias("total_amount")

    )

)



payment_analysis.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.payment_analysis"
)



print("Payment Analysis created")





# ============================================================
# FINAL CHECK
# ============================================================


print("Gold Tables Created")

spark.sql("""
SHOW TABLES IN ecommerce.gold
""").show()

Gold schema ready
Monthly Sales created
Customer Summary created
Product Performance created
Sales By Country created
Return Analysis created
Payment Analysis created
Gold Tables Created
+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|    gold|  analytics_category|      false|
|    gold|analytics_custome...|      false|
|    gold|analytics_executi...|      false|
|    gold|   analytics_payment|      false|
|    gold|   analytics_returns|      false|
|    gold|analytics_revenue...|      false|
|    gold|analytics_top_cus...|      false|
|    gold|analytics_top_pro...|      false|
|    gold|    customer_summary|      false|
|    gold|       monthly_sales|      false|
|    gold|    payment_analysis|      false|
|    gold| product_performance|      false|
|    gold|     return_analysis|      false|
|    gold|    sales_by_country|      false|
+--------+--------------------+-----------+

